
# Agentic Multimodal Demo (Notebook)
Lightweight, agentic pipeline you can run step-by-step:
- **SeriesBuilder**: ordered people/things with dates (kings, CEOs, etc.) → SDXL portraits → poster
- **MapBuilder**: regions/points (Europe, US states, Canadian provinces, South America) → flags/icons → map

Repo layout assumption:
```
/notebooks/agentic_multimodal/agentic-multimodal.ipynb      # this notebook (root)
/src/agentic_multimodal #supporting modules
/results/agentic_multimodal    # output data
```


In [1]:

# --- Optional installs (uncomment as needed) ---
# %pip install langgraph langchain pydantic diffusers transformers accelerate
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# %pip install pillow geopandas shapely requests matplotlib bs4 pyproj
# %pip install ipywidgets
# Note: On Macs, torch install line will differ (MPS). See PyTorch docs.


#### Load all the relevant code into a registry

In [2]:
import pathlib as p
from agentic_multimodal.core.config import CACHE, RESULTS, SRC
from agentic_multimodal.schemas.artifacts import PosterSpec, PosterItem, ImageAsset
from agentic_multimodal.services.registry import make_registry
from agentic_multimodal.skills.adapters import name_year_pairs, countries_to_map_spec
from agentic_multimodal.skills.image_gen import generate_person_images
from agentic_multimodal.skills.gen_poster_renderer import compose_poster_spec
import sys
from support import tree_markdown
import os, re

ROOT = p.Path("..").resolve().parent
MANIFEST = ROOT / "results" / "agentic_multimodal" / "manifest.jsonl"
reg = make_registry(ROOT)

BUILD_TEASER = False
BUILD_POTUS = True
BUILD_EUROPE = False
BUILD_NOBEL_PHYSICS = False



In [3]:

for dir in [CACHE, RESULTS]:
    dir.mkdir(parents=True, exist_ok=True)  # ensure directory exists
    if str(dir) not in sys.path:
        sys.path.insert(0, str(dir))

# Show code tree from source:
display(tree_markdown(SRC))

```agentic_multimodal/
├── assets
│   ├── ne
│   │   ├── ne_110m_admin_0_countries.geojson
│   │   └── ne_50m_admin_0_countries.geojson
│   └── prompts
│       └── overrides.yaml
├── cache
├── core
│   ├── __init__.py
│   └── config.py
├── graphs
│   ├── __init__.py
│   ├── factory.py
│   ├── geo_flow.py
│   └── person_flow.py
├── schemas
│   ├── __init__.py
│   ├── artifacts.py
│   ├── entities.py
│   ├── messages.py
│   └── requests.py
├── services
│   ├── cache.py
│   ├── io_web_fetcher.py
│   ├── llm_factory.py
│   ├── registry.py
│   └── settings.py
├── skills
│   ├── adapters
│   │   ├── __init__.py
│   │   ├── countries_to_map.py
│   │   └── people_to_poster.py
│   ├── data
│   │   ├── __init__.py
│   │   ├── natural_earth.py
│   │   ├── wikidata_geo.py
│   │   ├── wikidata_search_label.py
│   │   ├── wikidata_series.py
│   │   └── wikidata_sparql.py
│   ├── geo
│   │   ├── __init__.py
│   │   ├── aliases.py
│   │   ├── europe_flags.py
│   │   └── subdivisions.py
│   ├── series
│   │   ├── __init__.py
│   │   ├── aliases.py
│   │   ├── award.py
│   │   └── positions.py
│   ├── __init__.py
│   ├── gen_map_renderer.py
│   ├── gen_poster_renderer.py
│   ├── image_gen.py
│   ├── image_prompts.py
│   ├── map_regions.py
│   └── research_people.py
├── __init__.py
└── notebook_utils.py
```

## Poster compositor (grid + captions)
### Build teaser image of a handful of presidents

In [4]:
# 3×2 teaser (names + all term ranges)

if BUILD_TEASER:
    reg = make_registry(ROOT)

    teaser_names = [
        "George Washington",
        "Abraham Lincoln",
        "John F. Kennedy",
        "Ronald Reagan",
        "Barack Obama",
        "Donald Trump",
    ]

    # 1) Look up term years from your series data
    people = reg.series.run("potus")

    def _y(s):
        if not s: return None
        s = s.lstrip("+")
        return s[:4] if len(s) >= 4 and s[:4].isdigit() else None

    def term_spans(person):
        # Build "YYYY–YYYY" or "YYYY–PT"; sort by start year
        spans = []
        for t in getattr(person, "terms", []):
            ys = _y(getattr(t, "start", None))
            ye = _y(getattr(t, "end", None))
            if ys or ye:
                spans.append((ys, ye))
        # sort by numeric start, then end
        spans.sort(key=lambda ab: (int(ab[0]) if (ab[0] and ab[0].isdigit()) else 9999,
                                int(ab[1]) if (ab[1] and ab[1].isdigit()) else 9999))
        # format with en dash
        out = []
        for ys, ye in spans:
            if ys and ye:
                out.append(f"{ys}–{ye}")
            elif ys and not ye:
                out.append(f"{ys}–PT")
            elif not ys and ye:
                out.append(f"–{ye}")
        return ", ".join(out)

    by_name = {p.name: p for p in people}

    # 2) Map names → existing portrait paths (first match per name)
    IMG_DIR = "artifacts/potus_terms_portraits"   # adjust if needed

    def _canon(s): return re.sub(r"\W+", "", s).lower()
    paths_by_name = {}
    for fn in sorted(os.listdir(IMG_DIR)):
        if not fn.lower().endswith((".png",".jpg",".jpeg",".webp")):
            continue
        name_part = re.sub(r"^\d+_", "", os.path.splitext(fn)[0])
        paths_by_name.setdefault(_canon(name_part), os.path.join(IMG_DIR, fn))

    def _portrait_for(name):
        p = paths_by_name.get(_canon(name))
        if not p:
            raise FileNotFoundError(f"No portrait found for '{name}' in {IMG_DIR}")
        return p

    # 3) Build items with NAME + merged term spans
    items = []
    for nm in teaser_names:
        label_lines = [nm]
        if nm in by_name:
            spans = term_spans(by_name[nm])
            if spans:
                label_lines.append(spans)
        label = "\n".join(label_lines)

        items.append(
            PosterItem(
                image=ImageAsset(id=_canon(nm), path=_portrait_for(nm), width=512, height=768),
                label=label
            )
        )

    # 4) 3 columns × 2 rows; export sized for LinkedIn (landscape-ish)
    spec_teaser = PosterSpec(
        title="U.S. Presidents — Agentic Poster (Teaser)",
        grid_cols=3,
        items=items
    )

    out_teaser = "artifacts/linkedin/potus_teaser_3x2_terms_1620x1080.jpg"
    os.makedirs(os.path.dirname(out_teaser), exist_ok=True)

    # Slightly larger captions help on mobile; tune if your renderer exposes these knobs
    compose_poster_spec(
        spec_teaser,
        outpath=out_teaser,
        out_format="JPEG",
        out_quality=88,
        max_long_side=1620,          # ~1620×1080 export
        # If your renderer supports it, these help readability:
        # caption_scale=2.2,
        # line_gap_px=8,
        # stroke_width_px=3,
    )

    print("Teaser saved:", out_teaser)



In [5]:
if BUILD_POTUS:
    reg = make_registry(ROOT)
    people = reg.series.run("potus")
    pairs = name_year_pairs(people, mode="per_term")
    print(pairs)
    #paths = generate_person_images(pairs, outdir="artifacts/potus_terms_portraits")

    #spec  = reg.adapters.people_per_term(people, title="U.S. Presidents — Terms", image_paths=paths, cols=6)
    #reg.render.poster(spec, outpath="artifacts/poster_presidents_terms.webp")

[('George Washington', '1789'), ('John Adams', '1797'), ('Thomas Jefferson', '1801'), ('James Madison', '1809'), ('James Monroe', '1817'), ('John Quincy Adams', '1825'), ('Andrew Jackson', '1829'), ('Martin Van Buren', '1837'), ('John Tyler', '1841'), ('William Henry Harrison', '1841'), ('James K. Polk', '1845'), ('Zachary Taylor', '1849'), ('Millard Fillmore', '1850'), ('Franklin Pierce', '1853'), ('James Buchanan', '1857'), ('Abraham Lincoln', '1861'), ('Andrew Johnson', '1865'), ('Ulysses S. Grant', '1869'), ('Rutherford B. Hayes', '1877'), ('Chester A. Arthur', '1881'), ('James A. Garfield', '1881'), ('Grover Cleveland', '1885'), ('Benjamin Harrison', '1889'), ('Grover Cleveland', '1893'), ('William McKinley', '1897'), ('Theodore Roosevelt', '1901'), ('William Howard Taft', '1909'), ('Woodrow Wilson', '1913'), ('Warren G. Harding', '1921'), ('Calvin Coolidge', '1923'), ('Herbert Hoover', '1929'), ('Franklin D. Roosevelt', '1933'), ('Harry S. Truman', '1945'), ('Dwight D. Eisenhower

In [6]:
if 1: #BUILD_NOBEL_PHYSICS:
    laureates = reg.series.run("nobel_physics")
    pairs = name_year_pairs(laureates, mode="per_person_auto")
#    paths = generate_person_images(pairs, outdir="artifacts/nobel_physics_portraits")
#    spec  = reg.adapters.people_per_person(laureates, title="Nobel Prize in Physics — Laureates", image_paths=paths, cols=6)
#    reg.render.poster(spec, outpath="artifacts/poster_nobel_physics.webp")
    print(laureates)

[Person(qid='Q35149', name='Wilhelm Röntgen', image_url='http://commons.wikimedia.org/wiki/Special:FilePath/Roentgen2.jpg', terms=[OfficeTerm(start='1901-01-01', end=None)]), Person(qid='Q41688', name='Hendrik Lorentz', image_url='http://commons.wikimedia.org/wiki/Special:FilePath/H%20A%20Lorentz%20%28Nobel%29.jpg', terms=[OfficeTerm(start='1902-01-01', end=None)]), Person(qid='Q79000', name='Pieter Zeeman', image_url='http://commons.wikimedia.org/wiki/Special:FilePath/Pieter%20Zeeman.jpg', terms=[OfficeTerm(start='1902-01-01', end=None)]), Person(qid='Q41269', name='Henri Becquerel', image_url='http://commons.wikimedia.org/wiki/Special:FilePath/Paul%20Nadar%20-%20Henri%20Becquerel.jpg', terms=[OfficeTerm(start='1903-01-01', end=None)]), Person(qid='Q7186', name='Marie Curie', image_url='http://commons.wikimedia.org/wiki/Special:FilePath/Marie%20Curie%20%281900%29%20%28cropped%29.jpg', terms=[OfficeTerm(start='1903-01-01', end=None)]), Person(qid='Q37463', name='Pierre Curie', image_ur

## MapBuilder: regions + capitals + flags

In [7]:
if BUILD_EUROPE:
    reg = make_registry(ROOT)
    countries = reg.geo.run("europe_countries_flags")

    spec = countries_to_map_spec(
        reg.geo.run("europe_countries_flags"),
        title="Europe — Flags at Capitals",
        region_key="europe_flags",
    )

    reg.render.map(
        spec,
        outdir="artifacts/maps",
        size=(2200, 1320),
        marker_px=42,              
        show_labels=True,        
        show_country_names=True,
        show_capital_names=False,
        min_flag_separation_px=48,
        min_label_separation_px=96,
        max_labels=None,
    )
    print("Saved:", spec.path)


In [8]:
import re
from agentic_multimodal.skills.data.wikidata_sparql import WikidataSPARQL
from agentic_multimodal.schemas.entities import Person  # your model

_QID_RE = re.compile(r"Q\d+$")

def fix_missing_labels(people: list[Person]) -> list[Person]:
    # 1) collect QIDs where name looks like "Q###"
    missing = [p.qid for p in people if _QID_RE.fullmatch(p.name or "")]
    if not missing:
        return people

    # 2) batch fetch English labels
    vals = " ".join(f"wd:{q}" for q in missing)
    q = f"""
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?item ?label WHERE {{
      VALUES ?item {{ {vals} }}
      ?item rdfs:label ?label .
      FILTER(LANGMATCHES(LANG(?label), "en"))
    }}
    """
    rows = WikidataSPARQL().run(q)
    by_qid = { r["item"]["value"].rpartition("/")[2]: r["label"]["value"] for r in rows }

    # 3) return NEW Person instances (pydantic v2: model_copy(update=...))
    fixed = []
    for p in people:
        if _QID_RE.fullmatch(p.name or ""):
            nm = by_qid.get(p.qid, p.name)
            fixed.append(p.model_copy(update={"name": nm}))
        else:
            fixed.append(p)
    return fixed


In [9]:
people = reg.series.run("potus")
people = fix_missing_labels(people)


In [10]:
prov = reg.series._providers["potus"]
print("alias ->", type(prov).__name__, getattr(prov, "_fixed", {}))
print("series default language:", reg.series.language)

# sanity: what base and params will run?
base = getattr(prov, "base", prov)  # PreconfiguredProvider wraps a base
print("base:", type(base).__name__)
print("params:", {**getattr(prov, "_fixed", {})})


alias -> PreconfiguredProvider {'position_qids': ['Q11696']}
series default language: [AUTO_LANGUAGE],en
base: PositionsProvider
params: {'position_qids': ['Q11696']}


In [11]:
rows = reg.series._providers["potus"].base.fetch(reg.series.sparql, position_qids=["Q11696"], language="[AUTO_LANGUAGE],en")
print(rows[0])
# Expect to see 'name' after you patch; right now you likely only see 'person', 'personLabel', etc. or just 'person'


qid='Q23' name='George Washington' image_url='http://commons.wikimedia.org/wiki/Special:FilePath/Gilbert%20Stuart%20Williamstown%20Portrait%20of%20George%20Washington.jpg' terms=[OfficeTerm(start='1789-04-30', end='1797-03-04')]
